In [57]:
import pandas as pd
df = pd.read_csv('merged_df.csv')

# df = df[:1000]

C:\Users\bscag\AppData\Local\Temp\ipykernel_53776\515031801.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('merged_df.csv')


In [58]:
gendered_terms = ['skirt','bonnet','corset','gown','skirts','dress']

In [59]:
import re

# def contains_gendered_term(terms, gendered_terms):
#     if not isinstance(terms, list):
#         return False
#     pattern = r'\b(?:' + '|'.join(map(re.escape, gendered_terms)) + r')\b'
#     joined = ' '.join(terms)
#     return bool(re.search(pattern, joined, flags=re.IGNORECASE))

def contains_gendered_term(term, gendered_terms):
    if term in gendered_terms:
        return True


df['has_gendered_term'] = df['term'].apply(lambda x: contains_gendered_term(x, gendered_terms))


In [60]:
df

,Unnamed: 0,book_id,mention_id,term,sentence,start_idx,end_idx,sentence_start_idx,sentence_end_idx,adjectives,character_id,gender,has_gendered_term
0,0,wu.89097998645.clean,6971477,sheath,I set her down \nas a New England woman the fi...,119,125,173434,173598,[],2947.0,he/him/his,None
1,1,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],282.0,she/her,None
2,2,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],281.0,he/him/his,None
3,3,wu.89097998645.clean,6971479,cloak,"While I was mourning over my cloak, she \ncame...",29,34,174093,174336,[],2947.0,he/him/his,None
4,4,wu.89097998645.clean,6971480,coat,"While I was mourning over my cloak, she \ncame...",163,167,174093,174336,[],186.0,she/her,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
329372,329372,yale.39002088679528.clean,15208824,clothing,But instantly it seemed \nto me that the voice...,122,130,896937,897183,['bare'],NaN,NaN,None
329373,329373,yale.39002088679528.clean,15208826,garments,"And they answered that the other women also, \...",152,160,899605,899838,[],NaN,NaN,None
329374,329374,yale.39002088679528.clean,15208827,linen,And what \nthou and the other women thought to...,91,96,900787,900912,[],NaN,NaN,None
329375,329375,yale.39002088679528.clean,15208828,raiment,And what \nthou and the other women thought to...,97,104,900787,900912,[],NaN,NaN,None


In [61]:
# df.loc[df['has_gendered_term'] == True & df['gender']== 'he/him/his']
df.loc[(df['has_gendered_term'] == True) & (df['gender'] == 'he/him/his')]


,Unnamed: 0,book_id,mention_id,term,sentence,start_idx,end_idx,sentence_start_idx,sentence_end_idx,adjectives,character_id,gender,has_gendered_term
39,39,wu.89097998645.clean,6971491,bonnet,"She began to untie her bonnet, \nsaying: \n“I'...",23,29,186092,186184,[],0.0,he/him/his,True
40,40,wu.89097998645.clean,6971491,bonnet,"She began to untie her bonnet, \nsaying: \n“I'...",23,29,186092,186184,[],1277.0,he/him/his,True
56,56,wu.89097998645.clean,6971505,gown,if the patriarch \nIsaac sat enjoying his news...,65,69,200764,201177,[],281.0,he/him/his,True
59,59,wu.89097998645.clean,6971505,gown,if the patriarch \nIsaac sat enjoying his news...,65,69,200764,201177,[],3103.0,he/him/his,True
60,60,wu.89097998645.clean,6971505,gown,if the patriarch \nIsaac sat enjoying his news...,65,69,200764,201177,[],314.0,he/him/his,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
320240,320240,yale.39002030973185.clean,3121100,skirts,I \nhave shown this person the inside of my ha...,94,100,12014,12219,['white'],0.0,he/him/his,True
320241,320241,yale.39002030973185.clean,3121100,skirts,I \nhave shown this person the inside of my ha...,94,100,12014,12219,['white'],961.0,he/him/his,True
320331,320331,yale.39002030973185.clean,3121127,dress,It was \nprobably because I had always spoken ...,354,359,19328,19788,[],996.0,he/him/his,True
320344,320344,yale.39002030973185.clean,3121137,gown,The great boulders thrust themselves \nthrough...,104,108,29114,29223,['ragged'],0.0,he/him/his,True


In [ ]:
df.loc[
    (df['has_gendered_term']) & (df['gender'] == 'he/him/his'),
    'label'
] = 'incorrect'


In [52]:
def mask_terms(sentence, terms):
    if not isinstance(terms, list):
        return sentence
    for t in terms:
        pattern = r'\b' + re.escape(t) + r'\b'
        sentence = re.sub(pattern, '[MASKED_TERM]', sentence, flags=re.IGNORECASE)
    return sentence

df['masked_sentence'] = df.apply(lambda row: mask_terms(row['sentence'], row['term']), axis=1)


In [53]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
df['embedding'] = df['masked_sentence'].apply(lambda x: model.encode(x))


In [54]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

X = np.vstack(df['embedding'])
y = (df['label'] == 'incorrect').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=500, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       1.00      0.86      0.92       189
           1       0.29      1.00      0.45        11

    accuracy                           0.86       200
   macro avg       0.64      0.93      0.69       200
weighted avg       0.96      0.86      0.90       200

ROC-AUC: 0.953102453102453


In [55]:
df['incorrect_proba'] = clf.predict_proba(np.vstack(df['embedding']))[:, 1]


In [56]:
suspect_candidates = df[df['incorrect_proba'] > 0.7].sort_values('incorrect_proba', ascending=False)
